# JARVIS-VLA leftover mouth

Qwen2-VL chat template on `CraftJarvis/JarvisVLA-Qwen2-VL-7B`. Pick kernel **jarvis-vqa**. This is not the MineStudio play loop.

Frames in `screenshot/` come from the official `minecraft-vla-sft` **valid** split. If that folder is empty:

`source .venv/bin/activate && python download_screenshots.py`

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from IPython.display import display, Markdown, clear_output
from PIL import Image
import ipywidgets as widgets
from qwen_mouth import JarvisQwenMouth

WEIGHTS = ROOT / "weights" / "JarvisVLA-Qwen2-VL-7B"
SHOT_DIR = ROOT / "screenshot"
SHOTS = sorted(
    p for p in SHOT_DIR.glob("*")
    if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
)
print("weights", WEIGHTS.exists(), WEIGHTS)
print("screenshots", len(SHOTS), SHOT_DIR)
if not SHOTS:
    raise FileNotFoundError(
        "screenshot/ is empty. In the jarvis-vqa venv: python download_screenshots.py"
    )

In [ ]:
mouth = JarvisQwenMouth(WEIGHTS)
print("loaded", mouth.weights)

## What it sees

Pick a screenshot. Re-run the ask cells after you change the dropdown.

In [ ]:
image_path = SHOTS[0]
frame = Image.open(image_path).convert("RGB")
picker = widgets.Dropdown(
    options=[(p.name, str(p)) for p in SHOTS],
    value=str(image_path),
    description="screenshot",
    layout=widgets.Layout(width="90%"),
    style={"description_width": "initial"},
)
preview = widgets.Output()

def _on_pick(change):
    global image_path, frame
    image_path = Path(change["new"])
    frame = Image.open(image_path).convert("RGB")
    with preview:
        clear_output(wait=True)
        display(frame)
        print(image_path.name, frame.size)

picker.observe(_on_pick, names="value")
display(widgets.VBox([picker, preview]))
_on_pick({"new": picker.value})

In [ ]:
question = "what are you looking at?"
out = mouth.ask(frame, question)
display(Markdown("**Reply**"))
display(Markdown(out["reply"] or "_(empty)_"))
print("verdict:", out["verdict"])
print("frame:", image_path.name)
print("log frame:", out["frame"])

Ask anything else about the same frame. `usable` = leftover mouth still talks. `empty` / `garbage` = log that SFT ate it.

In [ ]:
question = "If I asked you to mine a tree, what would you do first?"
out = mouth.ask(frame, question)
display(Markdown(out["reply"] or "_(empty)_"))
print("verdict:", out["verdict"])
print("frame:", image_path.name)